# SciQLop DSP Module

SciQLop provides a built-in DSP (Digital Signal Processing) module that works
directly with `SpeasyVariable` objects. All functions preserve metadata (name,
units, axes) and handle data gaps automatically.

This tutorial shows how to use DSP functions inside virtual products to build
live, interactive signal-processing pipelines.

<div align="center">
<img src="https://github.com/SciQLop/SciQLop/raw/main/SciQLop/resources/icons/SciQLop.png"/>
</div>

## 1. Low-pass filtering with a tunable cutoff

Wrap a Butterworth filter in a `%%vp` cell with a knob for the cutoff frequency.
The filtered signal updates live as you change the cutoff or pan the time range.

In [ ]:
%%vp --path "dsp_examples/filtered_B" --debug --start "2015-11-18T02:14:30" --stop "2015-11-18T03:34:00"
import speasy as spz
import numpy as np
from SciQLop.user_api import dsp
from scipy.signal import butter
from typing import Annotated
from SciQLop.user_api.knobs import Knob
from speasy.core import datetime64_to_epoch

def filtered_B(
    start: float, stop: float,
    cutoff: Annotated[float, Knob(min=0.01, max=2.0, step=0.01,
                                  label="Cutoff", unit="Hz")] = 0.5,
) -> Vector["Bx", "By", "Bz"]:
    b = spz.get_data("cda/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2", start, stop)
    if b is None:
        return None
    t = datetime64_to_epoch(b.time)
    vals = b.values[:, :3]
    sos = butter(4, cutoff, fs=16.0, output="sos").astype(vals.dtype)
    return dsp.arrays.sosfiltfilt(t, vals, sos)

## 2. Plotting original vs filtered side by side

Use the fluent API to compare the raw signal with the filtered virtual product.

In [ ]:
from SciQLop.user_api.plot import fluent

panel = (fluent.new_panel()
    .plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")
    .subplot()
    .plot("dsp_examples/filtered_B")
    .time_range("2015-11-18T02:14:30", "2015-11-18T03:34:00"))

## 3. Rolling statistics — envelope detection

Use `rolling_mean` and `rolling_std` to build a virtual product that shows
the signal envelope (mean ± std). The smoothing window is a tunable knob.

In [ ]:
%%vp --path "dsp_examples/B_magnitude_envelope" --debug --start "2015-11-18T02:14:30" --stop "2015-11-18T03:34:00"
import speasy as spz
import numpy as np
from SciQLop.user_api import dsp
from typing import Annotated
from SciQLop.user_api.knobs import Knob
from speasy.core import datetime64_to_epoch

def B_magnitude_envelope(
    start: float, stop: float,
    window: Annotated[int, Knob(min=8, max=512, step=8, label="Window")] = 64,
) -> MultiComponent["|B|", "mean", "mean+std", "mean-std"]:
    b = spz.get_data("cda/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2", start, stop)
    if b is None:
        return None
    t = datetime64_to_epoch(b.time)
    b_mag = np.sqrt(np.sum(b.values[:, :3] ** 2, axis=1, keepdims=True))
    t_mean, mean_vals = dsp.arrays.rolling_mean(t, b_mag, window)
    t_std, std_vals = dsp.arrays.rolling_std(t, b_mag, window)
    n = len(t_mean)
    raw = b_mag[:n].flatten()
    m = mean_vals.flatten()
    s = std_vals.flatten()
    return t_mean, np.column_stack([raw, m, m + s, m - s])

## 4. Live spectrogram with tunable FFT parameters

`dsp.spectrogram` returns 2D data (time × frequency) suitable for color map plots.
Wrap it in a `%%vp` to get a live spectrogram with tunable window size and overlap.

In [ ]:
%%vp --path "dsp_examples/Bx_spectrogram" --debug --start "2015-11-18T02:14:30" --stop "2015-11-18T03:34:00"
import speasy as spz
from SciQLop.user_api import dsp
from typing import Annotated, Literal
from SciQLop.user_api.knobs import Knob

def Bx_spectrogram(
    start: float, stop: float,
    window_size: Annotated[int, Knob(min=64, max=1024, step=64, label="FFT window")] = 256,
    overlap: Annotated[int, Knob(min=0, max=512, step=32, label="Overlap")] = 128,
    window: Literal["hann", "hamming", "blackman", "bartlett", "boxcar"] = "hann",
) -> Spectrogram:
    b = spz.get_data("cda/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2", start, stop)
    if b is None:
        return None
    overlap = min(overlap, window_size - 1)
    segs = dsp.spectrogram(b, col=0, window_size=window_size, overlap=overlap, window=window)
    if not segs:
        return None
    s = segs[0]
    return s.time, s.axes[1].values, s.values

## 5. Field magnitude

Compute |B| from the GSE vector components (the product includes a 4th column
with the pre-computed magnitude, but here we show the explicit calculation).

In [ ]:
%%vp --path "dsp_examples/B_magnitude" --debug --start "2015-11-18T02:14:30" --stop "2015-11-18T03:34:00"
import speasy as spz
import numpy as np
from speasy.core import datetime64_to_epoch

def B_magnitude(start: float, stop: float) -> Scalar["|B| (nT)"]:
    b = spz.get_data("cda/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2", start, stop)
    if b is None:
        return None
    t = datetime64_to_epoch(b.time)
    b_mag = np.sqrt(np.sum(b.values[:, :3] ** 2, axis=1, keepdims=True))
    return t, b_mag

## 6. Multi-panel overview

Combine all the DSP-powered VPs into a single multi-panel view.

In [ ]:
from SciQLop.user_api.plot import fluent

overview = (fluent.new_panel()
    .plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")
    .subplot()
    .plot("dsp_examples/filtered_B")
    .subplot()
    .plot("dsp_examples/B_magnitude")
    .subplot()
    .plot("dsp_examples/Bx_spectrogram")
    .log_y()
    .time_range("2015-11-18T02:14:30", "2015-11-18T03:34:00"))

## 7. Using DSP outside of virtual products

You can also call DSP functions directly on fetched data for one-off analysis.

In [ ]:
import speasy as spz
from SciQLop.user_api import dsp
from datetime import datetime

b = spz.get_data(
    "cda/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2",
    datetime(2015, 11, 18, 2, 14, 30),
    datetime(2015, 11, 18, 3, 34, 0)
)

# Split into continuous segments
segments = dsp.split_segments(b)
print(f"{len(segments)} segment(s)")

# Per-segment FFT
fft_results = dsp.fft(b)
freqs, mag = fft_results[0]
print(f"FFT: {len(freqs)} frequency bins, range {freqs[1]:.4f}–{freqs[-1]:.2f} Hz")

## API reference

All functions accept a `SpeasyVariable` and return a `SpeasyVariable`
(preserving metadata), except where noted.

| Function | Description | Returns |
|---|---|---|
| `dsp.filtfilt(data, coeffs)` | Zero-phase FIR filter | Same shape |
| `dsp.sosfiltfilt(data, sos)` | Zero-phase IIR (SOS) filter | Same shape |
| `dsp.fir_filter(data, coeffs)` | Per-segment FIR | Same shape |
| `dsp.iir_sos(data, sos)` | Per-segment IIR | Same shape |
| `dsp.interpolate_nan(data)` | Fill NaN runs | Same shape |
| `dsp.rolling_mean(data, window)` | Gap-aware rolling mean | Same shape |
| `dsp.rolling_std(data, window)` | Gap-aware rolling std | Same shape |
| `dsp.resample(data, target_dt=)` | Uniform resampling | New time axis |
| `dsp.fft(data)` | Per-segment FFT | `list[(freq, mag)]` |
| `dsp.spectrogram(data, col=, window_size=)` | Time-freq analysis | `list[SpeasyVariable]` (2D) |
| `dsp.reduce(data, op)` | Collapse columns (`'norm'`, `'mean'`, `'sum'`) | 1D |
| `dsp.reduce_axes(data, shape, axes, op=)` | Arbitrary axis reduction | Variable |
| `dsp.split_segments(data)` | Detect gaps, split | `list[SpeasyVariable]` |

For raw numpy arrays, use `dsp.arrays.<function>(x, y, ...)`.